In [7]:
"""
Threads user info pipeline (EnsembleData) — schema-fixed version

What it does:
1) Reads CSV: accounts_threads_userID.csv (must include a user id column)
2) Calls /threads/user/info for each user id
3) Saves raw JSON to: Threads/user_info/<user_id>.json
4) Loads all JSON files in that folder
5) Parses json["data"] into a table
6) Saves parsed table to: Threads/user_info.csv

Setup:
- pip install pandas requests
- Either:
  (A) export ENSEMBLEDATA_TOKEN="YOUR-TOKEN-HERE"
  (B) or set os.environ["ENSEMBLEDATA_TOKEN"] inside your notebook
"""

from __future__ import annotations

import os
import re
import json
import time
import logging
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests

In [8]:
# -----------------------
# Config
# -----------------------
ROOT = "https://ensembledata.com/apis"
ENDPOINT = "/threads/user/info"
TOKEN = os.getenv("ENSEMBLEDATA_TOKEN", "vKLIay8miZKARJKU")
INPUT_CSV = Path("Threads/threads_user_search.csv")
OUT_DIR = Path("Threads") / "user_info"   # change to Path("Threads") / "user info" if you want a space

SLEEP_SECS = 0.35
TIMEOUT_SECS = 60
OVERWRITE_EXISTING = False


# -----------------------
# Helpers
# -----------------------
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def safe_filename(name: str, max_len: int = 180) -> str:
    name = str(name).strip()
    name = re.sub(r"[^\w\-\.]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return (name or "unknown")[:max_len]

def to_int_or_none(x: Any) -> Optional[int]:
    try:
        if x is None:
            return None
        return int(x)
    except Exception:
        return None

def safe_get(d: Any, path: List[Any], default=None):
    cur = d
    for k in path:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        elif isinstance(cur, list) and isinstance(k, int) and 0 <= k < len(cur):
            cur = cur[k]
        else:
            return default
    return cur

def extract_bio_text(d: Dict[str, Any]) -> Optional[str]:
    """
    From your sample:
    text_app_biography.text_fragments.fragments[*].plaintext
    """
    fragments = safe_get(d, ["text_app_biography", "text_fragments", "fragments"], default=None)
    if not isinstance(fragments, list):
        return None
    parts = []
    for f in fragments:
        if isinstance(f, dict) and isinstance(f.get("plaintext"), str):
            parts.append(f["plaintext"])
    return "".join(parts) if parts else None

def extract_hd_profile_pic_urls(d: Dict[str, Any]) -> Optional[str]:
    """
    From your sample:
    hd_profile_pic_versions is a list of dicts with url/width/height.
    We'll store URLs joined by | for convenience.
    """
    versions = d.get("hd_profile_pic_versions")
    if not isinstance(versions, list):
        return None
    urls = []
    for v in versions:
        if isinstance(v, dict) and isinstance(v.get("url"), str):
            urls.append(v["url"])
    return "|".join(urls) if urls else None

def extract_bio_links(d: Dict[str, Any]) -> Optional[str]:
    """
    bio_links is a list of dicts with url etc.
    We'll store URLs joined by |.
    """
    bio_links = d.get("bio_links")
    if not isinstance(bio_links, list):
        return None
    urls = []
    for bl in bio_links:
        if isinstance(bl, dict) and isinstance(bl.get("url"), str):
            urls.append(bl["url"])
    return "|".join(urls) if urls else None

In [9]:
# -----------------------
# Step 1: Scrape and save raw JSONs
# -----------------------
def scrape_user_info(user_id: Any, token: str) -> Dict[str, Any]:
    url = ROOT + ENDPOINT
    params = {"id": user_id, "token": TOKEN}
    res = requests.get(url, params=params, timeout=TIMEOUT_SECS)
    res.raise_for_status()

    raw_json = res.json()
    return {
        "_meta": {
            "scrape_ts": utc_now_iso(),
            "endpoint": ENDPOINT,
            "query_id": str(user_id),
            "http_status": res.status_code,
            "request_url": res.url,
        },
        "data": raw_json,  # note: raw_json itself contains {"data": {...}}
    }

def run_scrape_from_csv() -> None:

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(INPUT_CSV)
    id_col = 'user_id'

    logging.info("Loaded %d rows from %s (id column: %s)", len(df), INPUT_CSV, id_col)

    for i, row in df.iterrows():
        user_id_raw = row[id_col]
        if pd.isna(user_id_raw):
            continue

        # keep as string for filenames; pass raw to API (int-like strings are fine)
        user_id_str = str(user_id_raw).strip()
        if not user_id_str or user_id_str.lower() == "nan":
            continue

        out_path = OUT_DIR / f"{safe_filename(user_id_str)}.json"

        if out_path.exists() and not OVERWRITE_EXISTING:
            logging.info("[%d/%d] Exists, skipping: %s", i + 1, len(df), out_path.name)
            continue

        try:
            logging.info("[%d/%d] Scraping user/info for id=%s", i + 1, len(df), user_id_str)
            wrapper = scrape_user_info(user_id=user_id_str, token=TOKEN)
            out_path.write_text(json.dumps(wrapper, ensure_ascii=False, indent=2), encoding="utf-8")
            time.sleep(SLEEP_SECS)

        except requests.HTTPError as e:
            err_wrapper = {
                "_meta": {
                    "scrape_ts": utc_now_iso(),
                    "endpoint": ENDPOINT,
                    "query_id": user_id_str,
                    "error": f"HTTPError: {e}",
                },
                "data": None,
            }
            out_path.write_text(json.dumps(err_wrapper, ensure_ascii=False, indent=2), encoding="utf-8")
            logging.exception("HTTP error for id=%s", user_id_str)

        except Exception as e:
            err_wrapper = {
                "_meta": {
                    "scrape_ts": utc_now_iso(),
                    "endpoint": ENDPOINT,
                    "query_id": user_id_str,
                    "error": f"Exception: {e}",
                },
                "data": None,
            }
            out_path.write_text(json.dumps(err_wrapper, ensure_ascii=False, indent=2), encoding="utf-8")
            logging.exception("Unexpected error for id=%s", user_id_str)

In [4]:
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

    # 1) Scrape user info results (raw JSON files)
    run_scrape_from_csv()

INFO: Loaded 7 rows from Threads/threads_user_search.csv (id column: user_id)
INFO: [1/7] Exists, skipping: 20269764.json
INFO: [2/7] Scraping user/info for id=43109246
INFO: [3/7] Exists, skipping: 13460080.json
INFO: [4/7] Scraping user/info for id=306787899
INFO: [5/7] Scraping user/info for id=2568684
INFO: [6/7] Scraping user/info for id=359256305
INFO: [7/7] Scraping user/info for id=1090677444


In [5]:
# -----------------------
# Step 2: Parse JSONs into a table
# -----------------------
def parse_all_jsons_to_table(out_csv: Path = Path("Threads") / "threads_user_info.csv") -> pd.DataFrame:
    if not OUT_DIR.exists():
        raise FileNotFoundError(f"Folder not found: {OUT_DIR}. Run scraping first.")

    rows: List[Dict[str, Any]] = []

    for fp in sorted(OUT_DIR.glob("*.json")):
        wrapper = json.loads(fp.read_text(encoding="utf-8"))
        meta = wrapper.get("_meta", {}) if isinstance(wrapper, dict) else {}

        scrape_ts = meta.get("scrape_ts")
        query_id = meta.get("query_id") or fp.stem
        http_status=meta.get("http_status") or 0

        raw_payload = wrapper.get("data") if isinstance(wrapper, dict) else None
        # raw_payload should be {"data": {...}} per your sample
        data_obj = raw_payload.get("data") if isinstance(raw_payload, dict) else None

        if not isinstance(data_obj, dict):
            rows.append({
                "query_id": query_id,
                "http_status": http_status,
                #"pk": None,
                "user_id": None,
                "username": None,
                "full_name": None,
                "is_verified": None,
                #"text_post_app_is_private": None,
                "follower_count": None,
                "profile_pic_url": None,
                "hd_profile_pic_urls": None,
                "biography": None,
                "text_biography_plain": None,
                "bio_links": None,
                #"eligible_for_text_app_activation_badge": None,
                #"hide_text_app_activation_badge_on_text_app": None,
                "is_threads_only_user": None,
                #"transparency_label": None,
                "scrape_ts": scrape_ts,
                "source_file": fp.name,
            })
            continue

        rows.append({
            "query_id": query_id,
            "http_status": http_status,
            #"pk": data_obj.get("pk"),
            "user_id": data_obj.get("id"),
            "username": data_obj.get("username"),
            "full_name": data_obj.get("full_name"),
            "is_verified": data_obj.get("is_verified"),
            "text_post_app_is_private": data_obj.get("text_post_app_is_private"),
            "follower_count": to_int_or_none(data_obj.get("follower_count")),
            "profile_pic_url": data_obj.get("profile_pic_url"),
            "hd_profile_pic_urls": extract_hd_profile_pic_urls(data_obj),
            "biography": data_obj.get("biography"),
            "text_biography_plain": extract_bio_text(data_obj),
            "bio_links": extract_bio_links(data_obj),
            #"eligible_for_text_app_activation_badge": data_obj.get("eligible_for_text_app_activation_badge"),
            #"hide_text_app_activation_badge_on_text_app": data_obj.get("hide_text_app_activation_badge_on_text_app"),
            "is_threads_only_user": data_obj.get("is_threads_only_user"),
            #"transparency_label": data_obj.get("transparency_label"),
            "scrape_ts": scrape_ts,
            "source_file": fp.name,
        })

    df_out = pd.DataFrame(rows)
    df_out.sort_values(["query_id"], inplace=True, na_position="last")

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(out_csv, index=False)

    logging.info("Saved parsed user info CSV: %s (%d rows)", out_csv, len(df_out))
    return df_out


In [6]:

# -----------------------
# Main
# -----------------------
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")


    # 2) Parse all saved JSONs into a table
    parse_all_jsons_to_table()


INFO: Saved parsed user info CSV: Threads/user_info.csv (7 rows)
